In [8]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm


In [9]:
# Load the dataset from the .npz file
def load_dataset(file_path):
    data = np.load(file_path)
    features = data['features']  # (N, 14)
    targets = data['targets']    # (N, 50, 2)
    return features, targets

# Example: Loading the dataset
features, targets = load_dataset("pose_traject_dataset_2000x50_0.npz")
print("Features shape:", features.shape)
print("Targets shape:", targets.shape)


Features shape: (100000, 14)
Targets shape: (100000, 50, 7)


In [10]:
# Gaussian Path Generator: Add noise to the trajectory data
def add_gaussian_noise(trajectory, noise_std=0.1):
    """Add Gaussian noise to the trajectory data"""
    noise = np.random.normal(0, noise_std, trajectory.shape)
    noisy_trajectory = trajectory + noise
    return noisy_trajectory


In [11]:
class ResidualTemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualTemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(out_channels)

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.bn2(x)
        return self.relu(x + residual)

class UNetFlowMatching(nn.Module):
    def __init__(self, input_dim=9, output_dim=2, num_steps=50, kernel_size=3):
        super(UNetFlowMatching, self).__init__()
        
        self.input_dim = input_dim
        self.num_steps = num_steps
        
        # Encoder: Downsample the temporal sequence
        self.enc1 = ResidualTemporalBlock(input_dim, 64, kernel_size)
        self.enc2 = ResidualTemporalBlock(64, 128, kernel_size)
        self.enc3 = ResidualTemporalBlock(128, 256, kernel_size)

        # Bottleneck: Midcoder, no downsampling
        self.bottleneck = ResidualTemporalBlock(256, 512, kernel_size)

        # Decoder: Upsample the temporal sequence
        self.dec1 = ResidualTemporalBlock(512, 256, kernel_size)
        self.dec2 = ResidualTemporalBlock(256, 128, kernel_size)
        self.dec3 = ResidualTemporalBlock(128, 64, kernel_size)

        # Final Convolution (vector field output)
        self.final_conv = nn.Conv1d(64, output_dim, kernel_size=kernel_size, padding=kernel_size//2)

    def forward(self, x):
        # Encoding
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)

        # Bottleneck
        x4 = self.bottleneck(x3)

        # Decoding
        x5 = self.dec1(x4)
        x6 = self.dec2(x5)
        x7 = self.dec3(x6)

        # Final output: vector field for trajectory evolution
        out = self.final_conv(x7)
        return out


In [7]:
# Loss function for Flow Matching (MSE loss)
criterion = nn.MSELoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training loop
def train_flow_matching_model(model, dataloader, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for batch_idx, (inputs, targets) in enumerate(dataloader):
            optimizer.zero_grad()

            # Add Gaussian noise to the targets (creating a noisy trajectory)
            noisy_targets = add_gaussian_noise(targets)

            # Forward pass: get vector field predictions from model
            outputs = model(inputs.permute(0, 2, 1))  # Permute to match (batch, channels, time)

            # Compute loss: compare predicted vector field with noisy trajectory
            loss = criterion(outputs, noisy_targets.permute(0, 2, 1))  # Permute to match (batch, channels, time)
            running_loss += loss.item()

            # Backpropagation
            loss.backward()
            optimizer.step()

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(dataloader):.4f}")

# Start training
train_flow_matching_model(model, train_dataloader, num_epochs=10)


NameError: name 'model' is not defined

In [6]:
# ODE Solver to evolve the trajectory using the learned vector field
def ode_solver(vector_field, initial_state, num_steps, dt=0.1):
    """
    Solves the ODE using the Euler method for trajectory generation.
    Args:
        vector_field: The predicted vector field from the model (U-Net).
        initial_state: The initial state of the trajectory (x0, y0).
        num_steps: Number of steps in the trajectory.
        dt: Time step for integration.
    Returns:
        trajectory: The generated trajectory after integrating the vector field.
    """
    trajectory = [initial_state]
    state = initial_state

    for t in range(num_steps):
        # Compute the change in state based on the vector field
        field = vector_field[:, t, :]  # Extract the vector field at timestep t
        dx = field * dt  # Multiply the vector field by the time step (Euler method)
        state = state + dx
        trajectory.append(state)

    return torch.stack(trajectory)

# Example of using the ODE solver:
initial_state = torch.tensor([0.0, 0.0])  # Starting position
vector_field = model(torch.tensor(features))  # Get vector field from the model
trajectory = ode_solver(vector_field, initial_state, num_steps=50)


In [7]:
# Visualization: Plot predicted vs true trajectory
def plot_trajectory(true_trajectory, predicted_trajectory):
    plt.figure(figsize=(8, 6))
    plt.plot(true_trajectory[:, 0], true_trajectory[:, 1], label="True Trajectory", color='g')
    plt.plot(predicted_trajectory[:, 0], predicted_trajectory[:, 1], label="Predicted Trajectory", color='r', linestyle='--')
    plt.xlabel('X Position (m)')
    plt.ylabel('Y Position (m)')
    plt.title('True vs Predicted Trajectory')
    plt.legend()
    plt.show()

# Evaluate and visualize predictions
predicted_trajectory = trajectory.numpy()
true_trajectory = targets[0]  # Use the first target trajectory for visualization
plot_trajectory(true_trajectory, predicted_trajectory)


In [8]:
# Adjust hyperparameters as needed
model = UNetFlowMatching(input_dim=9, output_dim=2, num_steps=50, kernel_size=5)  # Change kernel size

# Adjust learning rate for the optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train with the new settings
train_flow_matching_model(model, train_dataloader, num_epochs=20)
